# 02_03 Search three ways: which method finds the right review?

Once documents are vectors, "which review is about this?" becomes "which vector is closest?". This notebook
answers that question three ways over Kittiwake's 400 reviews: TF-IDF cosine similarity, **BM25** (the
ranking search engines use), and **sentence embeddings** (vectors a model learned, which see meaning). Each
one gets a query the others get wrong.

**How this notebook works.** The same rhythm as Lab 01:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-02-turning-words-into-numbers", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'bm25s': 'bm25s',
           'sentence_transformers': 'sentence-transformers',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import csv
import json
import math
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nlpcheck import ask, guess, reveal, check_02_03

reviews3 = ["This movie is very scary and long",
            "This movie is not scary and is slow",
            "This movie is spooky and good"]
kw = list(csv.DictReader(open("data/kittiwake_reviews.csv")))
print(len(kw), "Kittiwake app reviews;", kw[0]["stars"], "stars:", kw[0]["text"])
os.makedirs("out", exist_ok=True)

## 1. Recall

**r5.** Which words get a high IDF? (a) words in few documents, (b) words in many documents,
(c) long words

**r6.** What IDF does scikit-learn give a word that appears in every document? (a number)

In [ ]:
ask("r5", "")
ask("r6", "")

## 2. Cosine similarity

The **cosine similarity** of two vectors is the cosine of the angle between them: 1 when they point the
same way, 0 when they share nothing. It is the dot product divided by both lengths, so a long review and a
short one about the same thing still score high. `TfidfVectorizer` already made every row length 1, so here
the cosine is the dot product.

In [ ]:
tv = TfidfVectorizer()
T = tv.fit_transform(reviews3)
print(np.round(cosine_similarity(T), 3))
print("dot product of rows 1 and 2:", round(float(T[0].multiply(T[1]).sum()), 3))

Predict: how similar are "this movie is scary" and "this movie is not scary" as bags of words?
(a number between 0 and 1)

In [ ]:
guess("scary_not_scary", None)

In [ ]:
pair = ["this movie is scary", "this movie is not scary"]
reveal("scary_not_scary", round(float(cosine_similarity(CountVectorizer().fit_transform(pair))[0, 1]), 3))

`0.894`: opposite meanings, four shared words out of five. Now the other direction: predict the TF-IDF
similarity of "a spooky film" and "a scary movie", which mean the same thing.

In [ ]:
guess("spooky_scary_tfidf", None)

In [ ]:
pair = ["a spooky film", "a scary movie"]
reveal("spooky_scary_tfidf", round(float(cosine_similarity(TfidfVectorizer().fit_transform(pair))[0, 1]), 3))

`0.0`. They share no column (`a` is dropped by the token pattern), and every word is its own axis. Section 5
comes back to this pair with a method that can see it.

## 3. Search with TF-IDF

Fit the vectorizer on the reviews, transform the query **with the same vectorizer** so it lands in the same
columns, sort by cosine similarity:

In [ ]:
texts = [r["text"] for r in kw]
ktv = TfidfVectorizer()
K = ktv.fit_transform(texts)

def top(scores, k=3):
    best = np.argsort(scores)[::-1][:k]
    return [(kw[i]["review_id"], round(float(scores[i]), 3), kw[i]["text"][:60]) for i in best]

def tfidf_scores(query):
    return cosine_similarity(ktv.transform([query]), K).ravel()

for hit in top(tfidf_scores("dropped calls")):
    print(hit)

## 4. Search with BM25

**BM25** is the ranking function behind most search boxes, and the first stage of many retrieval systems that
feed documents to a language model. It keeps TF-IDF's idea, rare words count for more, and changes two
things, which page 5 of the chapter draws:

- **repetition saturates**: a word's contribution grows with its count but levels off (`k1 = 1.5`);
- **length is compared with the average**: a longer-than-average review is discounted (`b = 0.75`), and the
  review's other words do not dilute anything.

`bm25s` is a small, fast BM25 library. Index once, then score any query against every review:

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # a progress-bar notice, not an error
import bm25s

bm25 = bm25s.BM25()                        # k1 = 1.5, b = 0.75, the usual defaults
bm25.index(bm25s.tokenize(texts, stopwords=None, show_progress=False), show_progress=False)

def bm25_scores(query):
    words = bm25s.tokenize([query], stopwords=None, show_progress=False, return_ids=False)[0]
    return np.asarray(bm25.get_scores(words))

for hit in top(bm25_scores("dropped calls")):
    print(hit)

Same winners as TF-IDF here, on a different scale: BM25 scores are not between 0 and 1, only their order
matters. Now a query where the two disagree. Predict: for **"support hold"**, will TF-IDF and BM25 put the
same review first? (`True` or `False`)

In [ ]:
guess("bm25_same_top", None)

In [ ]:
q = "support hold"
t, b = tfidf_scores(q), bm25_scores(q)
print("TF-IDF:", top(t))
print("BM25  :", top(b))
reveal("bm25_same_top", kw[t.argmax()]["text"] == kw[b.argmax()]["text"])

Different winners, and BM25's is the better answer. It ranks "Waited 80 minutes on hold for support." first, a
hold-time complaint; TF-IDF prefers "Support is friendly but the hold times are long", a mixed review. Both
contain *support* and *hold* once. Cosine divides by the length of the whole TF-IDF vector, so the first
review's rare words (*waited*, *80*, *minutes*) dilute its score, while the second's common words (*is*, *but*,
*the*, *are*) barely do. BM25 scores only the query's words, adjusted for the review's length.

## 5. Search with sentence embeddings

Both methods so far match **words**. A **sentence embedding** is a list of numbers a neural model produces for a
whole text, trained on a very large amount of text so that texts with similar meanings get nearby vectors, even
with no word in common. This one comes from `ibm-granite/granite-embedding-small-english-r2`, already in the
image: 384 numbers per text. Chapter 5 and Lab 05 show how such vectors are learned; here you use them.

A real search system embeds its documents **once**, when they are added, and at search time embeds only the
query. This lab does the same: when your session started, `warm_embeddings.py` embedded the 400 reviews in
the background and saved the vectors to `out/review_embeddings.npz`. The cell below loads them when they
are there and match the reviews, and otherwise computes them, which takes **about forty seconds**. Importing
the library and loading the model takes **about fifteen seconds** either way. The cell says which happened.

In [ ]:
from sentence_transformers import SentenceTransformer
from warm_embeddings import MODEL, review_vectors

model = SentenceTransformer(MODEL)             # ibm-granite/granite-embedding-small-english-r2
E, how = review_vectors(model, texts)          # one row of 384 numbers per review
print(E.shape, how)

def embed_scores(query):
    return E @ model.encode([query], normalize_embeddings=True)[0]   # cosine, since every row has length 1

Back to the pair TF-IDF scored `0.0`. Predict the embedding similarity of "a spooky film" and "a scary movie".

In [ ]:
guess("spooky_scary_embed", None)

In [ ]:
def embed_sim(a, b):
    va, vb = model.encode([a, b], normalize_embeddings=True)
    return round(float(va @ vb), 3)

reveal("spooky_scary_embed", embed_sim("a spooky film", "a scary movie"))
print("spooky / invoice          :", embed_sim("spooky", "invoice"))
print("scary / not scary         :", embed_sim("this movie is scary", "this movie is not scary"))

`0.893`: the meaning is there at last. The next two lines keep you honest. *spooky* against *invoice* still
scores `0.745`, because this model's scores crowd into the upper range, so an embedding score means something
only compared with other scores for the same query. And "scary" against "not scary" scores `0.951`, higher
than the bag of words gave it: embeddings see topic and meaning, and a single *not* still barely moves them.
Negation stays hard until the models of Labs 10 to 12.

Now the query that separates the methods. **"the phone company took too much money"** shares almost no word
with the reviews about being overcharged. Predict which method ranks an overcharging review first:
`"tfidf"`, `"bm25"` or `"embeddings"`.

In [ ]:
guess("money_query_winner", None)

In [ ]:
q = "the phone company took too much money"
for name, fn in (("tfidf", tfidf_scores), ("bm25", bm25_scores), ("embeddings", embed_scores)):
    print(f"{name:10}", top(fn(q), k=1))
reveal("money_query_winner", "embeddings")

TF-IDF and BM25 both return "Switching with the eSIM took two minutes": it matches *took*, the only rare query
word the reviews use. The embeddings return "They overcharged me ...". And the reverse happens too: try
`"waited ages on the phone"` in the three methods. BM25 finds the hold-time reviews through *waited*; the
embeddings drift to reviews about calls. That is why production search often runs both, **hybrid** search, and
merges the two rankings.

## 6. Your turn

Save the three reviews closest to **"no signal at home"** by **TF-IDF cosine**, using `top` and
`tfidf_scores`.

In [ ]:
top3 = []   # YOUR CODE HERE: the three review ids, most similar first

json.dump({"query": "no signal at home", "top3": top3}, open("out/02_03_search.json", "w"), indent=1)
check_02_03();

## 7. Exit ticket

**x3.** Why do "a spooky film" and "a scary movie" score 0 with TF-IDF? (a) they are too short,
(b) TF-IDF removes adjectives, (c) they share no column, and counting cannot see synonyms

In [ ]:
ask("x3", "")

Explain it back: for one query of your own, say which of the three methods you would trust, and why.

*Your explanation:* 